In [1]:
import random
import numpy as np
import librosa
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt
import csv
import os

MAX_SAMPLES = 20000
SPLIT = 0.8


In [2]:
class MyDataset(Dataset):
    def __init__(self, train=True, max_samples=MAX_SAMPLES):
        super().__init__()
        self.spoof_files = []
        self.bonafide_files = []
        self.audio_path = "flac_T/flac_T/" if train else "flac_D/flac_D/"  # path to the directory containing the audio files
        metadata_file = "ASVspoof5.train.metadata.txt" if train else "ASVspoof5.dev.metadata.txt"
        with open(metadata_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                label = parts[-1]  # last word is "spoof" or "bonafide"
                if label == "spoof":
                    self.spoof_files.append(parts[1]+".flac")   # use full line instead if you want the whole row
                elif label == "bonafide":
                    self.bonafide_files.append(parts[1]+".flac")
        random.seed(42)  # optional: makes the result reproducible
        self.spoof_ds = random.sample(self.spoof_files, min(max_samples, len(self.spoof_files)))
        self.bonafide_ds = random.sample(self.bonafide_files, min(max_samples, len(self.bonafide_files)))

    def __getitem__(self, idx):
        sample_rate = 16000

        if idx < len(self.spoof_ds):
            full_path = self.audio_path + self.spoof_ds[idx]
            label = 1
        else:
            full_path = self.audio_path + self.bonafide_ds[idx - len(self.spoof_ds)]
            label = 0

        waveform, sr = librosa.load(full_path, sr=None, mono=True)

        if sr != sample_rate:
            waveform = librosa.resample(waveform, orig_sr=sr, target_sr=sample_rate)

        mel = librosa.feature.melspectrogram(
            y=waveform,
            sr=sample_rate,
            n_fft=1024,
            hop_length=256,
            n_mels=80,
            fmin=0.0,
            fmax=8000
        )
        log_mel = librosa.power_to_db(mel, ref=np.max)
        log_mel = log_mel[np.newaxis, :, :]  # [1, n_mels, time]

        return torch.tensor(log_mel, dtype=torch.float32), torch.tensor(label)

    def __len__(self):
        return len(self.spoof_ds) + len(self.bonafide_ds)


In [3]:

class MyModel(nn.Module):
    def __init__(self, n_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            
            nn.MaxPool2d(2),  # reduce time/freq
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            
            nn.AdaptiveAvgPool2d((1, 1))  # works for different spectrogram sizes
        )
        self.classifier = nn.Linear(128, n_classes)

    def forward(self, x):
        # x shape: [batch, 1, n_mels, time]
        x = self.net(x)
        x = x.squeeze(-1).squeeze(-1)
        return self.classifier(x)

In [4]:
def pad_batch(batch):
    """
    Batch format: list of tuples (spec, label)
    Each spec has shape [1, n_mels, T]
    We pad along the time dimension to the max T in the batch.
    """
    specs = [x for x, _ in batch]
    labels = torch.tensor([y for _, y in batch], dtype=torch.long)

    max_T = max(s.shape[-1] for s in specs)
    padded_specs = []

    for s in specs:
        pad = max_T - s.shape[-1]
        if pad > 0:
            s = F.pad(s, (0, pad))
        padded_specs.append(s)

    return torch.stack(padded_specs), labels

In [5]:
def train_model():
    dataset = MyDataset()

    # Simple split: train/validation
    total = len(dataset)
    train_size = int(SPLIT * total)
    val_size = total - train_size

    train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(
        train_ds,
        batch_size=4,
        shuffle=True,
        collate_fn=pad_batch,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=4,
        shuffle=False,
        collate_fn=pad_batch,
        num_workers=2,
        pin_memory=True
    )

    model = MyModel()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    best_val_loss = float("inf")

    train_losses = []
    val_losses = []
    epochs = 1

    for epoch in range(epochs):
        print("=" * 60)
        print(f"Epoch {epoch+1}/{epochs}")
        print("=" * 60)
        model.train()
        running_loss = 0.0
        correct = 0
        total_seen = 0

        for specs, labels in tqdm(train_loader):
            specs = specs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(specs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total_seen += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_losses.append(train_loss)
        train_acc = correct / total_seen
        print(f"Epoch {epoch+1}/{epochs} | Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f}")

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss_total = 0.0
        val_batches = 0
        with torch.no_grad():
            for specs, labels in tqdm(val_loader):
                specs = specs.to(device)
                labels = labels.to(device)

                logits = model(specs)
                loss = criterion(logits, labels)
                val_loss_total += loss.item()
                val_batches += 1

                preds = logits.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_loss = val_loss_total / max(1, val_batches)
        val_losses.append(val_loss)
        val_acc = val_correct / val_total
        print(f"Validation loss: {val_loss:.4f} | Validation acc: {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_mel_cnn_spoof_bonafide.pt")
            print("Saved best model with lower validation loss.")

    print("Training complete.")
    print(f"Best validation loss: {best_val_loss:.4f}")

    return train_losses, val_losses


In [6]:
train_model()

Epoch 1/1


  0%|          | 0/7760 [00:00<?, ?it/s]/home/pietro/miniconda3/envs/ptorch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/pietro/miniconda3/envs/ptorch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 7760/7760 [16:33<00:00,  7.81it/s]


Epoch 1/1 | Train loss: 0.2495 | Train acc: 0.9053


  0%|          | 0/1940 [00:00<?, ?it/s]/home/pietro/miniconda3/envs/ptorch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/pietro/miniconda3/envs/ptorch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 1940/1940 [03:57<00:00,  8.17it/s]

Validation loss: 0.0791 | Validation acc: 0.9934
Saved best model with lower validation loss.
Training complete.
Best validation loss: 0.0791


([0.24948539862408634], [0.07913041883976361])

In [10]:
def test_model(
    max_samples=MAX_SAMPLES,
    batch_size=4,
    checkpoint_path="best_mel_cnn_spoof_bonafide.pt",
    hits_path="test_hits.csv",
    misses_path="test_misses.csv",
):
    """Evaluate the saved model and save correctly/incorrectly classified files."""
    dataset = MyDataset(train=False, max_samples=max_samples)
    if len(dataset) == 0:
        raise ValueError("The development dataset is empty.")
    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(f"Model checkpoint not found: {checkpoint_path}")

    val_loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=pad_batch,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MyModel().to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()

    all_labels = []
    all_predictions = []
    all_probabilities = []
    with torch.no_grad():
        for specs, labels in tqdm(val_loader, desc="Testing"):
            logits = model(specs.to(device))
            probabilities = torch.softmax(logits, dim=1).cpu().numpy()
            all_labels.extend(labels.numpy().tolist())
            all_predictions.extend(probabilities.argmax(axis=1).tolist())
            all_probabilities.extend(probabilities.tolist())

    y_true = np.asarray(all_labels, dtype=np.int64)
    y_pred = np.asarray(all_predictions, dtype=np.int64)
    probabilities = np.asarray(all_probabilities, dtype=np.float64)
    class_names = {0: "bonafide", 1: "spoof"}
    file_names = dataset.spoof_ds + dataset.bonafide_ds
    file_paths = [os.path.join(dataset.audio_path, name) for name in file_names]

    if len(file_paths) != len(y_true):
        raise RuntimeError("Prediction count does not match the development file list.")

    class_metrics = {}
    for class_id, class_name in class_names.items():
        true_positive = int(np.sum((y_true == class_id) & (y_pred == class_id)))
        false_positive = int(np.sum((y_true != class_id) & (y_pred == class_id)))
        false_negative = int(np.sum((y_true == class_id) & (y_pred != class_id)))
        precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
        recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        class_metrics[class_name] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }

    metrics = {
        "accuracy": float(np.mean(y_true == y_pred)),
        "macro_precision": float(np.mean([m["precision"] for m in class_metrics.values()])),
        "macro_recall": float(np.mean([m["recall"] for m in class_metrics.values()])),
        "macro_f1": float(np.mean([m["f1"] for m in class_metrics.values()])),
        "per_class": class_metrics,
    }

    fieldnames = [
        "file_path",
        "true_label",
        "predicted_label",
        "confidence",
        "bonafide_probability",
        "spoof_probability",
    ]
    with open(hits_path, "w", newline="", encoding="utf-8") as hits_file, open(
        misses_path, "w", newline="", encoding="utf-8"
    ) as misses_file:
        hits_writer = csv.DictWriter(hits_file, fieldnames=fieldnames)
        misses_writer = csv.DictWriter(misses_file, fieldnames=fieldnames)
        hits_writer.writeheader()
        misses_writer.writeheader()
        for index, (true_label, predicted_label) in enumerate(zip(y_true, y_pred)):
            row = {
                "file_path": file_paths[index],
                "true_label": class_names[int(true_label)],
                "predicted_label": class_names[int(predicted_label)],
                "confidence": float(probabilities[index, predicted_label]),
                "bonafide_probability": float(probabilities[index, 0]),
                "spoof_probability": float(probabilities[index, 1]),
            }
            (hits_writer if true_label == predicted_label else misses_writer).writerow(row)

    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"Macro precision: {metrics['macro_precision']:.4f}")
    print(f"Macro recall: {metrics['macro_recall']:.4f}")
    print(f"Macro F1: {metrics['macro_f1']:.4f}")
    for class_name, values in class_metrics.items():
        print(
            f"{class_name}: precision={values['precision']:.4f}, "
            f"recall={values['recall']:.4f}, F1={values['f1']:.4f}"
        )
    print(f"Correct predictions saved to: {hits_path}")
    print(f"Incorrect predictions saved to: {misses_path}")
    return metrics


test_metrics = test_model()

Testing:   0%|          | 0/10000 [00:00<?, ?it/s]/home/pietro/miniconda3/envs/ptorch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/pietro/miniconda3/envs/ptorch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7c1632ba4680>Exception ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7c1632ba4680>Traceback (most recent call last):

  File "/home/pietro/miniconda3/envs/ptorch/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1706, in __del__
Traceback (most recent call last):
  File "/home/pietro/miniconda3/

Accuracy: 0.7771
Macro precision: 0.8143
Macro recall: 0.7771
Macro F1: 0.7702
bonafide: precision=0.7061, recall=0.9492, F1=0.8098
spoof: precision=0.9225, recall=0.6049, F1=0.7307
Correct predictions saved to: test_hits.csv
Incorrect predictions saved to: test_misses.csv
